# Find linear dependence of tensors obtained with deltas 

Two deltas: $\delta_{ij} \delta_{kl} T_{ijklmn}$, etc

In [11]:
from carten.reduce import symmetrize_and_remove_trace, contract_with_delta, get_contraction_with_delta_rules
from carten.utils import check_symmetric, check_traceless
import torch
from torch import Tensor
import itertools

In [12]:
def check_linear_dependence(tensors:list[Tensor], 
                            factors:list[int], 
                            atol=1e-6, 
                            rtol=1e-6)->tuple[bool, list[int], Tensor]:
    """
    Check if the tensors are linearly dependent. 
    
    a_0 T_0 + a_1 T_1 + ... + a_n T_n = 0, 
    
    where a_i are the factors.
    
    Here, all possible combinations of factors are tried for each a_i.
    
    Args:
        tensors: the tensors to check 
        factors: possible non-zero factors, e.g.  [-2, -1, 1, 2]

    Returns:
        linear_dependence: True if the tensors are linearly dependent, False otherwise
        combination: the combination of factors that give the linear dependence
    """ 
    n = len(tensors)
    
    tensors = torch.stack(tensors, dim=-1)
    
    for fac in itertools.product(factors, repeat=n):
        fac = torch.tensor(fac).to(torch.float)
        sum_tensor = torch.sum(fac*tensors, dim=-1)
        if torch.allclose(sum_tensor, torch.zeros_like(sum_tensor), atol=atol, rtol=rtol):
            return True, fac, sum_tensor
        # else:
        #     print("Combination", fac, 'sum:', sum_tensor)
    
    return False, None, None
     
    

## Create a rank-6 tensor

In [13]:
torch.manual_seed(35)

t = torch.randn(3,3,3,3,3,3) 

## Select two pairs of indices (four) to contract with two deltas
## and get the natural tensors

In [14]:
rules = get_contraction_with_delta_rules(t.ndim, 2)
len(rules)

45

In [15]:
candidates = contract_with_delta(t, 2, rules)
natural_tensors = [symmetrize_and_remove_trace(c) for c in candidates]

len(natural_tensors)

45

In [16]:
for i, nt in enumerate(natural_tensors):
    if not check_symmetric(nt):
        print(f"Natural tensor {i} is not symmetric")
    if not check_traceless(nt):
        print(f"Natural tensor {i} is not traceless")

In [17]:
def check_one(natural_tensors, rules, num_selected, factors=[-1, 1, -2, 2],
                atol=1e-3, rtol=1e-3, 
              only_print_dependent=True
              ):
    """
    Args:
        natural_tensors: all natural tensors
        rules: all rules
        num_selected: number natural tensors to select to check linear dependence
        factors:  factors to try
    """
    all_selected = itertools.combinations(range(len(natural_tensors)), num_selected)
    
    for selected_index in all_selected:
        
        selected_rules = [rules[i] for i in selected_index] 
        selected_nts = [natural_tensors[i] for i in selected_index]
        dependent, combs, sums = check_linear_dependence(selected_nts, factors, atol=atol, rtol=rtol)
        
        if not only_print_dependent or dependent:
            print('='*40) 
            print('Linear dependence:', dependent)
            print('Rules:', selected_rules)
            print('Combination factors:', combs)
            print('Combined:', sums)
            if sums is not None:
                mx = sums.abs().max()
            else:
                mx = None
            print('max combined:', mx) 
        

## Show all candidate tensors

In [18]:
for i,r in enumerate(rules):
    print('candidate:', i, r)

candidate: 0 abcdef,cd,ef->ab
candidate: 1 abcdef,ce,df->ab
candidate: 2 abcdef,cf,de->ab
candidate: 3 abcdef,bd,ef->ac
candidate: 4 abcdef,be,df->ac
candidate: 5 abcdef,bf,de->ac
candidate: 6 abcdef,bc,ef->ad
candidate: 7 abcdef,bc,df->ae
candidate: 8 abcdef,bc,de->af
candidate: 9 abcdef,be,cf->ad
candidate: 10 abcdef,bf,ce->ad
candidate: 11 abcdef,bd,cf->ae
candidate: 12 abcdef,bd,ce->af
candidate: 13 abcdef,bf,cd->ae
candidate: 14 abcdef,be,cd->af
candidate: 15 abcdef,ad,ef->bc
candidate: 16 abcdef,ae,df->bc
candidate: 17 abcdef,af,de->bc
candidate: 18 abcdef,ac,ef->bd
candidate: 19 abcdef,ac,df->be
candidate: 20 abcdef,ac,de->bf
candidate: 21 abcdef,ae,cf->bd
candidate: 22 abcdef,af,ce->bd
candidate: 23 abcdef,ad,cf->be
candidate: 24 abcdef,ad,ce->bf
candidate: 25 abcdef,af,cd->be
candidate: 26 abcdef,ae,cd->bf
candidate: 27 abcdef,ab,ef->cd
candidate: 28 abcdef,ab,df->ce
candidate: 29 abcdef,ab,de->cf
candidate: 30 abcdef,ab,cf->de
candidate: 31 abcdef,ab,ce->df
candidate: 32 abcd

## Check the linear dependence 

In [19]:
# check_one(natural_tensors, rules, num_selected=45, factors=[-1, 1])


In [20]:
# select groups: ab, cd, ef 
nt_subset = []
rules_subset = []
for t, r in zip(natural_tensors, rules):
    ## 
    # #r = r.split('->')[-1]
    # if r in ['ab',
    #          'cd',
    #          'ef',
    #          ]:
    
    ##
    
    if r in [
    'abcdef,cd,ef->ab',
    'abcdef,af,de->bc',
    'abcdef,ab,ef->cd',
    'abcdef,af,bc->de',
    'abcdef,ab,cd->ef',
    'abcdef,bc,de->af'
    ]:
    ##  
    # if r in ['abcdef,cd,ef->ab',
    #         'abcdef,ab,ef->cd',
    #          'abcdef,ab,cd->ef',
    #          ]:
        nt_subset.append(t)
        rules_subset.append(r)

print(len(nt_subset))

check_one(nt_subset, rules_subset, num_selected=len(nt_subset), factors=[-1,-2,-3, 1,2,3])


6
